In [ ]:
import os
import pandas as pd

# from metabolights_utils.provider.study_provider import MetabolightsStudyProvider
from metabolights_utils.provider.ftp_repository import MetabolightsFtpRepository

# Select MetaboLights Studies

| Study ID | Focus | Samples (n) | Features (p) | Classification |
|----------|-------|-------------|--------------|----------------|
| MTBLS12968 | Polycystic Ovarian Syndrome (PCOS) | 149 | 486 | PCOS vs. healthy |
| MTBLS1 | Type 2 Diabetes | 132 | 220 | Diabetes vs. Healthy |
| MTBLS2 | Arabidopsis thaliana (plant) | 16 | 15272 | 2 Genotypes |



### **MTBLS12968**: PCOS preterm plasma metabolome

https://www.ebi.ac.uk/metabolights/editor/MTBLS12968/descriptors

*Xinyue HanPI*

This study investigated mid-trimester plasma metabolomic profiles in pregnant women to identify biomarkers associated with Polycystic Ovary Syndrome (PCOS) and preterm birth. A total of 149 participants were categorized into four groups: 32 with PCOS and preterm delivery, 34 with PCOS and term delivery, 42 without PCOS with preterm delivery, and 41 without PCOS with term delivery. Plasma samples were analyzed using targeted metabolomics via UHPLC-QTRAP MS/MS with HILIC and C18 chromatography. Multivariate statistical analyses including OPLS-DA were employed to identify significantly altered metabolites. The findings aim to provide insights into metabolic mechanisms underlying PCOS and preterm birth, potentially contributing to improved risk prediction strategies.


### **MTBLS1**: A metabolomic study of urinary changes in type 2 diabetes in human compared to the control group

https://www.ebi.ac.uk/metabolights/editor/MTBLS1/descriptors

*Reza M Salek, Jules L Griffin*

Type 2 diabetes mellitus is the result of a combination of impaired insulin secretion with reduced insulin sensitivity of target tissues. There are an estimated 150 million affected individuals worldwide, of whom a large proportion remains undiagnosed because of a lack of specific symptoms early in this disorder and inadequate diagnostics. In this study, NMR-based metabolomic analysis in conjunction with uni- and multivariate statistics was applied to examine the urinary metabolic changes in Human type 2 diabetes mellitus patients compared to the control group. The human population were un medicated diabetic patients who have good daily dietary control over their blood glucose concentrations by following the guidelines on diet issued by the American Diabetes Association. Note: This is part of a larger study, please refer to the original paper below.


### **MTBLS2**: Comparative LC/MS-based profiling of silver nitrate-treated Arabidopsis thaliana leaves of wild-type and cyp79B2 cyp79B3 double knockout plants

https://www.ebi.ac.uk/metabolights/editor/MTBLS2/descriptors

*Christoph Böttcher,  Steffen R Neumann*

Indole-3-acetaldoxime (IAOx) represents an early intermediate of the biosynthesis of a variety of indolic secondary metabolites including the phytoanticipin indol-3-ylmethyl glucosinolate and the phytoalexin camalexin (3-thiazol-2′-yl-indole). Arabidopsis thaliana cyp79B2 cyp79B3 double knockout plants are completely impaired in the conversion of tryptophan to indole-3-acetaldoxime and do not accumulate IAOx-derived metabolites any longer. Consequently, comparative analysis of wild-type and cyp79B2 cyp79B3 plant lines has the potential to explore the complete range of IAOx-derived indolic secondary metabolites.


<!-- ### **MTBLS8**: Growth control of the eukaryote cell: a systems biology study in yeast

https://www.ebi.ac.uk/metabolights/editor/MTBLS8/descriptors

*Juan I Castrillo,  Stephen G Oliver,  Warwick B Dunn,  Neil Swainston,  Juan I Castrillo,  Leo A Zeef,  David C Hoyle,  Nianshu Zhang,  Andrew Hayes,  David Cj Gardner,  Michael J Cornell,  June Petty,  Luke Hakes,  Leanne Wardleworth,  Bharat Rash,  Marie Brown,  Warwick B Dunn,  David Broadhurst,  Kerry O'Donoghue*

Metabolic control analysis is being exploited in a systems biology study of the eukaryotic cell. Using chemostat culture, we have measured the impact of changes in flux (growth rate) on the transcriptome, proteome, endometabolome and exometabolome of the yeast Saccharomyces cerevisiae. Each functional genomic level shows clear growth-rate-associated trends and discriminates between carbon-sufficient and carbon-limited conditions. Genes consistently and significantly upregulated with increasing growth rate are frequently essential and encode evolutionarily conserved proteins of known function that participate in many protein-protein interactions. In contrast, more unknown, and fewer essential, genes are downregulated with increasing growth rate; their protein products rarely interact with one another. A large proportion of yeast genes under positive growth-rate control share orthologs with other eukaryotes, including humans. Significantly, transcription of genes encoding components of the TOR complex (a major controller of eukaryotic cell growth) is not subject to growth-rate regulation. Moreover, integrative studies reveal the extent and importance of post-transcriptional control, patterns of control of metabolic fluxes at the level of enzyme synthesis, and the relevance of specific enzymatic reactions in the control of metabolic fluxes during cell growth. -->

In [ ]:
name2id = {
    "diabetes": "MTBLS1",
    "pcos": "MTBLS12968",  #  Y["control_vs_patient"] = Y["Factor Value[Group]"].str.split("-", n=1).str[0]
    "arabidopsis": "MTBLS2",
    "yeast": "MTBLS8",  # 126 samples × 383 metabolites ...cell growth control, not sure what the classes are (6?)
    # https://www.ebi.ac.uk/metabolights/editor/MTBLS8/descriptors
}

study_id = name2id["pcos"]

In [ ]:
# 1. Load the study using FTP repository
repo = MetabolightsFtpRepository()
study, messages = repo.load_study_model(study_id)

# The study files are downloaded to a local directory
study_path = os.path.join(repo.local_storage_root_path, study_id)
print(f"Study downloaded to: {study_path}")

# 2. Get the Sample Metadata (Response Vector)
if study.samples:
    samples_file = list(study.samples.keys())[0]
    sample_df = pd.read_csv(os.path.join(study_path, samples_file), sep="\t")
    sample_df = sample_df.set_index("Sample Name")
    print(f"Loaded samples from {samples_file}: {sample_df.shape}")
else:
    raise ValueError("No samples file found in the study")

# 3. Search for all MAF (Metabolite Assignment File) TSV files in the directory
# Look for files containing "MAF" or metabolite-related keywords
import glob

maf_files = []
for pattern in ["*MAF*.tsv", "*metabolite*.tsv", "*_maf_*.tsv"]:
    maf_files.extend(glob.glob(os.path.join(study_path, pattern)))

# Remove duplicates
maf_files = list(set(maf_files))

if maf_files:
    print(f"Found {len(maf_files)} MAF file(s):")
    for f in maf_files:
        print(f"  - {os.path.basename(f)}")

    # Load and concatenate all MAF files
    all_features = []

    for maf_path in maf_files:
        maf_filename = os.path.basename(maf_path)

        # Load the MAF file
        data_matrix = pd.read_csv(maf_path, sep="\t")
        print(f"\nMAF loaded from {maf_filename}: {data_matrix.shape}")

        # Find sample columns by matching with sample names
        sample_cols = [col for col in data_matrix.columns if col in sample_df.index]

        if sample_cols:
            # Extract intensity data for this MAF
            X_maf = data_matrix[sample_cols]

            # Get feature names from the specified column if available
            feature_col = "metabolite_identification"
            if feature_col in data_matrix.columns:
                # Use the feature column as index
                X_maf.index = data_matrix[feature_col].values
                print(f"  Using '{feature_col}' as feature names")
            else:
                print(f"  Warning: '{feature_col}' not found, using numeric index")

            print(f"  Extracted {len(sample_cols)} samples × {X_maf.shape[0]} features")
            all_features.append(X_maf)
        else:
            print(f"  No sample columns found in {maf_filename}")

    if all_features:
        # Concatenate all MAF data matrices (along features/rows)
        X_raw = pd.concat(all_features, axis=0)

        # Transpose so samples are rows and features are columns
        X_raw = X_raw.T
        X_raw.index.name = "Sample Name"

        # Check if data is available
        non_nan_count = X_raw.notna().sum().sum()
        if non_nan_count == 0:
            print(f"\n⚠️  WARNING: No intensity values found in MAF files!")
            print(
                f"This study ({study_id}) may not have processed quantification data available."
            )
            print(f"Try a different study ID that has complete data.")
            X, Y = None, None
        else:
            X = X_raw.loc[sample_df.index.intersection(X_raw.index)]
            Y = sample_df.loc[X.index]

            print(f"\n✓ Successfully loaded data:")
            print(f"  X: {X.shape[0]} samples × {X.shape[1]} metabolites")
            print(f"  Y: {Y.shape[0]} samples × {Y.shape[1]} metadata attributes")
            print(f"  Non-missing values: {non_nan_count:,}")
    else:
        print(f"No valid data found in MAF files")
        X, Y = None, None
else:
    print("No MAF files found in the study directory")
    X, Y = None, None

In [ ]:
# Display the loaded data
if X is not None and Y is not None:
    print("DATA SUMMARY")
    print("=" * 60)
    print(f"Samples: {X.shape[0]}")
    print(f"Metabolites: {X.shape[1]}")
    print(
        f"Missing values: {X.isna().sum().sum():,} ({100*X.isna().sum().sum()/(X.shape[0]*X.shape[1]):.1f}%)"
    )

    # Show experimental factors
    factor_cols = [col for col in Y.columns if "Factor" in col]
    if factor_cols:
        print(f"\nExperimental Factors:")
        for col in factor_cols:
            print(f"  {col}:")
            print(f"    {Y[col].value_counts().to_dict()}")
else:
    print("No data to display - please try a different study ID")

In [ ]:
Y.sample(10)

In [ ]:
X.info()

In [ ]:
# Y["control_vs_patient"] = Y["Factor Value[Group]"].str.split("-", n=1).str[0]
# Y["control_vs_patient"].value_counts()

In [ ]:
study_id

In [ ]:
Y

# Preprocess and save as CSV

In [ ]:
# map the study ID to response variable name to be extracted from Y
id2response = {
    "MTBLS1": "Factor Value[Metabolic syndrome]",
    "MTBLS12968": "control_vs_patient",
    "MTBLS2": "Factor Value[Genotype]",
}

# (binary) values from the response variable
# control is always 0
id2binary = {
    "MTBLS1": {
        "Control Group": 0,
        "diabetes mellitus": 1,
    },
    "MTBLS12968": {
        "HP": 0,
        "PCOS": 1,
    },
    "MTBLS2": {
        "Col-0": 0,
        "cyp79": 1,
    },
}

In [ ]:
# new_cols_dict = {i: f"feature_{i}" for i in X.columns}
# df = X.rename(columns=new_cols_dict)
df = X
df["response"] = Y[id2response[study_id]].map(id2binary[study_id])
df.info()

In [ ]:
df.to_csv(f"metabolights_{study_id}.csv", index=True)